# Phase 12 — Geometric-Lens-Karte (Token × Startlayer)

Die 2D-Fassung des Lens-Sweeps: Für jedes Prompt-Token und jede Tiefe wird
gemessen, wie der Druck am Tor reagiert, wenn dieses Token ab dort blind für
seinen Kontext ist. Blau = Druck fällt (das Token wird in dieser Tiefe
gelesen). Drei Karten plus Zahlen. Keine Generierung, ~2–4 min.

Selbstversorgend — **frische Runtime**, dann nur diese Zelle.

In [ ]:
# === Geometric-Lens-KARTE: Token x Startlayer =============================
# Die 2D-Fassung des Lens-Sweeps. Fuer jedes Prompt-Token t und jeden Voll-
# Attention-Layer l: Token t ist AB Layer l blind fuer seinen Kontext (nur
# Selbst-Attention), gemessen wird der Druck (Fremdschrift-Masse) am TOR -
# der Antwort-Position mit der groessten unterdrueckten Masse. Faellt der
# Druck, wird dieses Token in dieser Tiefe gelesen.
# Drei Karten: (1) Heatmap Token x Startlayer, Koeder-Spalten markiert,
# (2) Tiefenprofil der Koeder-Tokens gegen den Kontroll-Median, (3) wo der
# Druck ueber die Antwort-Positionen sitzt und wie er unter Blindheit kippt.
# Keine Generierung - nur Forwards, ~2-4 Minuten.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
SEED=0; STEP=4; NPOS=8
MASK_NPZ=(glob.glob("/content/drive/MyDrive/**/vocab_foreign_masks.npz",recursive=True) or [""])[0]
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- pure Logik (testbar) --------------------------------------
def blind_row(M,t,neg):
    """Zeile t der 4D-Maske: alles zu ausser Selbst-Attention (Kopie)"""
    M2=M.clone()
    if M2.dtype==torch.bool: M2[:,:,t,:]=False; M2[:,:,t,t]=True
    else:                    M2[:,:,t,:]=neg;   M2[:,:,t,t]=0
    return M2
def sample_tokens(lo,hi,step,must):
    """gleichmaessiges Raster ueber [lo,hi] plus Pflicht-Tokens, sortiert/eindeutig"""
    return sorted(set(list(range(lo,hi+1,step))+[t for t in must if lo<=t<=hi]))
def log_ratio(P,base,eps=1e-9):
    """log2(Druck / Baseline-Druck), robust gegen Nullen"""
    return np.log2(np.maximum(np.asarray(P,float),eps)/max(base,eps))
def top_drops(P,base,toks,l_idx,k=3):
    """die k Tokens mit dem staerksten Druckabfall in Layer-Zeile l_idx"""
    r=[(P[l_idx][j]/max(base,1e-9),toks[j]) for j in range(len(toks))]
    return sorted(r)[:k]
def tok_span(offs,c0,c1):
    return [i for i,(s,e) in enumerate(offs) if e>c0 and s<c1 and e>s]
# ---------------- Aufbau ----------------------------------------------------
try: model.set_attn_implementation("eager")
except Exception:
    try: model.config._attn_implementation="eager"
    except Exception: pass
FULL=sorted(int(re.fullmatch(r"model\.layers\.(\d+)\.self_attn",n).group(1))
            for n,_ in model.named_modules() if re.fullmatch(r"model\.layers\.(\d+)\.self_attn",n))
MODS=dict(model.named_modules())
ATTN={l:MODS["model.layers.%d.self_attn"%l] for l in FULL}
TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
EN_TAB=("| Service Name | Storage Limit |\n|---|---|\n"
 "| Google Drive | Google Drive offers 15 GB of free storage for every account. |")
pre=think_prefix(TAB,"")
enc=tokenizer(pre,return_offsets_mapping=True); IDS=enc["input_ids"]; L=len(IDS)
c0=len(SCAFF)+TAB.index("local name"); c1=c0+len("local name")
DEC=tok_span(enc["offset_mapping"],c0,c1); Q,K=DEC[-1],DEC[0]
UT=tok_span(enc["offset_mapping"],len(SCAFF),len(SCAFF)+len(TAB))
TOKS=sample_tokens(UT[0],UT[-1],STEP,[K-1,K,Q,Q+1])
print("Karte: %d Voll-Attention-Layer %s | %d Prompt-Tokens (Raster %d + Koeder-Umgebung)"
      %(len(FULL),FULL,len(TOKS),STEP))
print("  Koeder: K=%d %r | Q=%d %r"%(K,tokenizer.decode([IDS[K]]),Q,tokenizer.decode([IDS[Q]])))
HK={"tok":None,"start":None}
def mk_pre(lidx):
    def _pre(mod,args,kwargs):
        if HK["tok"] is None or lidx<HK["start"]: return None
        hs=kwargs.get("hidden_states", args[0] if (args and torch.is_tensor(args[0])) else None)
        if hs is None or hs.shape[1]<=1: return None
        am=kwargs.get("attention_mask",None)
        if not (torch.is_tensor(am) and am.dim()==4 and am.shape[-2]==hs.shape[1]): return None
        neg=torch.finfo(am.dtype).min if am.dtype.is_floating_point else False
        kwargs["attention_mask"]=blind_row(am,HK["tok"],neg)
        return (args,kwargs)
    return _pre
hooks=[ATTN[l].register_forward_pre_hook(mk_pre(l),with_kwargs=True) for l in FULL]
assert MASK_NPZ and os.path.exists(MASK_NPZ), "vocab_foreign_masks.npz nicht gefunden"
_z=np.load(MASK_NPZ); M_script=torch.tensor(_z["script"])
full=pre+EN_TAB
enc2=tokenizer(full,return_offsets_mapping=True); IDS2=enc2["input_ids"]
A0=next(i for i,(s,e) in enumerate(enc2["offset_mapping"]) if s>=len(pre) and e>s)
ids2=torch.tensor([IDS2],device=model.device)
@torch.no_grad()
def profile(tok,start):
    """Fremdschrift-Masse an den ersten NPOS Antwort-Positionen"""
    HK["tok"],HK["start"]=(None,None) if start is None else (tok,start)
    lg=model(input_ids=ids2).logits[0]
    HK["tok"],HK["start"]=None,None
    V=lg.shape[-1]; m=M_script.to(lg.device)
    if m.shape[0]<V: m=torch.cat([m,torch.zeros(V-m.shape[0],dtype=torch.bool,device=m.device)])
    m=m[:V]
    return np.array([float(torch.softmax(lg[A0-1+p].float(),-1)[m].sum()) for p in range(NPOS)])
try:
    base_prof=profile(None,None)
    GATE=int(np.argmax(base_prof))            # das Tor: Position mit der groessten Masse
    base=float(base_prof[GATE])
    print("  Baseline-Druckprofil ueber die Antwort: %s"%["%.4f"%v for v in base_prof])
    print("  -> Tor an Antwort-Position %d (Masse %.4f) - das ist der Readout der Karte"%(GATE,base))
    P=np.zeros((len(FULL),len(TOKS)))
    for i,l in enumerate(FULL):
        for j,t in enumerate(TOKS):
            P[i,j]=profile(t,l)[GATE]
    prof_blind=profile(Q,FULL[0])
finally:
    for h in hooks: h.remove()
    HK["tok"],HK["start"]=None,None
R=log_ratio(P,base)
# ---------------- Zahlen zur Karte ------------------------------------------
print("\n  Staerkster Druckabfall je Startlayer (Verhaeltnis zur Baseline):")
for i,l in enumerate(FULL):
    td=top_drops(P,base,TOKS,i,3)
    print("    ab L%-2d  %s"%(l," | ".join("Tok %d %r %.2fx"%(t,tokenizer.decode([IDS[t]]),r) for r,t in td)))
jQ=TOKS.index(Q); jK=TOKS.index(K)
ctrl=[j for j in range(len(TOKS)) if TOKS[j] not in (K-1,K,Q,Q+1)]
print("\n  Koeder vs. Kontrolle (medianes Verhaeltnis ueber alle Startlayer):")
print("    Q=%d %.2fx | K=%d %.2fx | Kontroll-Tokens %.2fx"
      %(Q,np.median(P[:,jQ])/base,K,np.median(P[:,jK])/base,np.median(P[:,ctrl])/base))
# ---------------- Karten ----------------------------------------------------
fig=plt.figure(figsize=(15,9))
gs=fig.add_gridspec(2,2,height_ratios=[1.35,1],hspace=.34,wspace=.22)
ax=fig.add_subplot(gs[0,:])
vm=float(np.nanmax(np.abs(R))) or 1.0
im=ax.imshow(R,aspect="auto",cmap="RdBu",vmin=-vm,vmax=vm,origin="lower")
ax.set_yticks(range(len(FULL))); ax.set_yticklabels(["ab L%d"%l for l in FULL],fontsize=9)
ax.set_xticks(range(len(TOKS)))
ax.set_xticklabels(["%d %s"%(t,tokenizer.decode([IDS[t]]).strip()[:8]) for t in TOKS],
                   rotation=90,fontsize=7.5)
for j in (jK,jQ):
    ax.add_patch(Rectangle((j-.5,-.5),1,len(FULL),fill=False,ec="#111",lw=2))
ax.set_title("Geometric-Lens-Karte: Token blind ab Layer -> Druck am Tor (Antwort-Position %d)\n"
             "ROT = Druck faellt (dieses Token wird dort gelesen) | blau = Druck steigt"%GATE,fontsize=11)
ax.set_xlabel("blind gemachtes Prompt-Token"); ax.set_ylabel("blind ab Voll-Attention-Layer")
plt.colorbar(im,ax=ax,fraction=.025,pad=.01,label="log2(Druck / Baseline)")
ax2=fig.add_subplot(gs[1,0])
ax2.axhline(base,ls="--",c="#888",lw=1,label="Baseline")
ax2.plot(FULL,P[:,jQ],"o-",c="#DC2626",lw=2,label="Q=%d %r"%(Q,tokenizer.decode([IDS[Q]])))
ax2.plot(FULL,P[:,jK],"s-",c="#EA9010",lw=2,label="K=%d %r"%(K,tokenizer.decode([IDS[K]])))
ax2.plot(FULL,np.median(P[:,ctrl],axis=1),"^-",c="#9CA3AF",lw=1.6,label="Kontroll-Tokens (median)")
ax2.set_yscale("log"); ax2.set_xlabel("blind ab Layer"); ax2.set_ylabel("Druck am Tor")
ax2.set_title("Tiefenprofil der Köder-Tokens",fontsize=10); ax2.legend(frameon=False,fontsize=8)
ax3=fig.add_subplot(gs[1,1])
x=np.arange(NPOS)
ax3.bar(x-.2,base_prof,.4,color="#2563EB",label="Baseline")
ax3.bar(x+.2,prof_blind,.4,color="#DC2626",label="Q blind ab L%d"%FULL[0])
ax3.set_yscale("log"); ax3.set_xticks(x); ax3.set_xlabel("Antwort-Position")
ax3.set_ylabel("Fremdschrift-Masse")
ax3.set_title("Wo der Druck sitzt — und wie er verschwindet",fontsize=10)
ax3.legend(frameon=False,fontsize=8)
plt.tight_layout(); plt.show()
MAP_RESULTS=dict(layers=FULL,tokens=TOKS,gate=GATE,base=base,P=P.tolist(),
                 base_prof=base_prof.tolist(),blind_prof=prof_blind.tolist())
